# ① 전처리 — 문서 파싱 · 청크 분할
담당: 전처리 · 코드: `rag/preprocessing/parsing.py`, `rag/preprocessing/chunking.py`

`config.yaml` 의 `paths.input`(ZIP 또는 폴더)에 있는 PDF · HWP · HWPX 를 읽어 본문 · 표 · (선택) 그림 설명을 뽑고, 청크로 나눕니다.

| 저장 파일 | 읽는 노트북 |
|---|---|
| `artifacts/chunks.jsonl` (청크) | 02 검색, 03 검증 |
| `artifacts/docs.jsonl` (문서 목록 — 발주기관 라우팅용) | 02 검색 |

In [ ]:
from rag.config import ARTIFACTS, ROOT, load_config
from rag.preprocessing.chunking import chunk_documents, chunk_summary, save_chunks
from rag.preprocessing.parsing import collect_inputs, parse_documents, parsing_summary

cfg = load_config()

## 설정
초기값은 `config.yaml`에서 읽습니다. 아래 변수의 오른쪽을 원하는 값으로 바꾸고 설정 셀부터 다시 실행하세요.
`OCR = True`이면 선택한 비전 모델로 그림을 텍스트로 바꿉니다(OpenAI API, 유료). 기본값은 `False`이며, 이때 그림을 API로 보내지 않습니다.
청킹 크기는 본문 글자 수 기준이며, 변경 후 01부터 후속 노트북을 다시 실행해야 합니다.


In [ ]:
OCR = False
OCR_MODEL = cfg["ocr"]["model"]                    # 예: "gpt-5-mini"
OCR_REASONING_EFFORT = cfg["ocr"]["reasoning_effort"]  # 예: "low"
OCR_MAX_TOKENS = cfg["ocr"]["max_tokens"]          # 출력·추론 토큰 한도
CHUNK_SIZE = cfg["chunking"]["chunk_size"]         # 예: 700

cfg["ocr"].update(model=OCR_MODEL, reasoning_effort=OCR_REASONING_EFFORT, max_tokens=OCR_MAX_TOKENS)
cfg["chunking"]["chunk_size"] = CHUNK_SIZE
cfg["ocr"], cfg["chunking"]


## 1. 입력 파일 모으기
ZIP(안의 ZIP 포함)을 `artifacts/raw/` 에 풀고 PDF · HWP · HWPX 목록을 만듭니다.

In [ ]:
files = collect_inputs(ROOT / cfg["paths"]["input"], ARTIFACTS / "raw")
len(files)

## 2. 파싱
문서마다 쪽 · 표 · 그림 수와 표지 감지 결과를 봅니다.

In [ ]:
docs = parse_documents(files, cfg["parsing"], ocr=OCR, ocr_cfg=cfg["ocr"])
parsing_summary(docs)

## 3. 청크 분할 · 저장

In [ ]:
chunks = chunk_documents(docs, cfg["chunking"])
save_chunks(docs, chunks, ARTIFACTS)
chunk_summary(chunks)